# 004 - Gold Load Revenue

This notebook aggregates silver-layer data into the **gold** layer, producing revenue summaries for analysis.

**Outputs:**
* `rentalproperty.gold.airbnb_per_listing` — total revenue per Airbnb listing
* `rentalproperty.gold.airbnb_per_postalcode` — total revenue per postal code (Airbnb)
* `rentalproperty.gold.rental_per_listing` — total revenue per rental listing
* `rentalproperty.gold.rental_per_postalcode` — total revenue per postal code (rentals)

**Sources:**
* `rentalproperty.silver.airbnb`
* `rentalproperty.silver.rentals`

Each gold table includes audit columns (`insertDateTime`, `updatedDateTime`) and is written in overwrite mode.

In [0]:
from pyspark.sql import functions as F

#set all variables to be used in this notebook
silver_rentals_table_name = f"rentalproperty.silver.rentals"
silver_airbnb_table_name = f"rentalproperty.silver.airbnb"
gold_airbnb_per_listing_table_name = f"rentalproperty.gold.airbnb_per_listing"
gold_airbnb_per_postalcode_table_name = f"rentalproperty.gold.airbnb_per_postalcode"
gold_rental_per_listing_table_name = f"rentalproperty.gold.rental_per_listing"
gold_rental_per_postalcode_table_name = f"rentalproperty.gold.rental_per_postalcode"

df_rental = spark.read.format("delta").table(silver_rentals_table_name)
df_airbnb = spark.read.format("delta").table(silver_airbnb_table_name)

In [0]:
df_airbnb_revenue_per_listing = (
     df_airbnb
    .groupBy("listingPk", "source", "latitude", "longitude")
    .agg(F.sum("price").alias("Revenue"))
    .select(
        "listingPk",
        "Revenue",
        F.col("source").alias("Source"),
        F.col("latitude").alias("Latitude"),
        F.col("longitude").alias("Longitude"),
        F.current_timestamp().alias("InsertedDateTime"),
        F.lit(None).cast("timestamp").alias("UpdatedDateTime"),
    )
)
#display(df_airbnb_revenue_per_listing)

In [0]:
df_airbnb_postalcode_revenue = (
    df_airbnb
    .groupBy("postalCode", "source")
    .agg(F.sum("price").alias("Revenue"))
    .select(
        F.col("postalCode").alias("PostalCode"),
        "Revenue",
        F.col("source").alias("Source"),
        F.current_timestamp().alias("InsertedDateTime"),
        F.lit(None).cast("timestamp").alias("UpdatedDateTime"),
    )
)
#display(df_airbnb_postalcode_revenue)

In [0]:
df_rental_revenue_per_listing = (
    df_rental
    .groupBy("listingPk", "source")
    .agg(F.sum("rentAmount").alias("Revenue"))
    .select(
        "listingPk",
        "Revenue",
        F.col("source").alias("Source"),
        F.current_timestamp().alias("InsertedDateTime"),
        F.lit(None).cast("timestamp").alias("UpdatedDateTime"),
    )
)

In [0]:
df_rental_revenue_per_postalcode = (df_rental.groupBy("postalCode", "source")
                .agg(F.sum("rentAmount").alias("Revenue"))
                .select(
                    F.col("postalCode").alias("PostalCode"),
                    "Revenue",
                    F.col("source").alias("Source"),
                    F.current_timestamp().alias("InsertedDateTime"),
                    F.lit(None).cast("timestamp").alias("UpdatedDateTime"),)
     )

In [0]:
df_airbnb_revenue_per_listing.write.mode("overwrite").saveAsTable(gold_airbnb_per_listing_table_name)
df_airbnb_postalcode_revenue.write.mode("overwrite").saveAsTable(gold_airbnb_per_postalcode_table_name)
df_rental_revenue_per_listing.write.mode("overwrite").saveAsTable(gold_rental_per_listing_table_name)
df_rental_revenue_per_postalcode.write.mode("overwrite").saveAsTable(gold_rental_per_postalcode_table_name)